In [1]:
# !git clone https://github.com/profcomff/chatbot-mark-api.git
!git clone --branch dev_fedor https://github.com/profcomff/chatbot-mark-api.git

Cloning into 'chatbot-mark-api'...
remote: Enumerating objects: 317, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 317 (delta 65), reused 105 (delta 39), pack-reused 169 (from 1)
Receiving objects: 100% (317/317), 3.53 MiB | 4.22 MiB/s, done.
Resolving deltas: 100% (117/117), done.


# Библиотеки


In [2]:
!pip install langchain transformers sentence-transformers -q
!pip install -U langchain-community -q
!pip install -qU "langchain-chroma>=0.1.2" -q
!pip install langchain_huggingface -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.

In [9]:
from tqdm import tqdm

import numpy as np
import pandas as pd

from transformers import XLMRobertaTokenizer, XLMRobertaModel
import torch

from langchain.schema import Document

from langchain_chroma import Chroma

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Функции/классы

In [4]:
import sys
sys.path.append("/content/chatbot-mark-api")

from nn.search import E5LangChainEmbedder

In [5]:
def safe_add_documents(vector_store, chunks, chroma_batch_size=1000):
    with tqdm(total=len(chunks), desc="Добавление в Chroma", unit="doc") as pbar:
        for i in range(0, len(chunks), chroma_batch_size):
            try:
                batch = chunks[i:i+chroma_batch_size]
                vector_store.add_documents(batch)
                pbar.update(len(batch))
            except Exception as e:
                if "Batch size" in str(e) and "greater than max" in str(e):
                    new_size = chroma_batch_size // 2
                    print(f"Ошибка: {e}. Уменьшаю размер батча до {new_size}")
                    return safe_add_documents(vector_store, chunks[i:], new_size)
                raise
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    print("Все документы успешно добавлены!")

# 0. Загрузка контекстов / скачивание модели


In [6]:
answers = pd.read_excel('/content/chatbot-mark-api/file/database_v2.xlsx')

display(answers.answer[0])
display(answers.head(2))

'Карта зачет. https://vk.com/wall-24234717_22977\nЭто ваш профсоюзный билет. С помощью этой карты вы можете получать скидки у полезных для студентов популярных брендов, участвовать в конкурсах и розыгрышах, а также посещать концерты и мероприятия. \nПолный перечень скидок есть в статье: vk.cc/bYSCNw.'

,Unnamed: 0,topic_name,answer,id
0,0,Карта зачет,Карта зачет. https://vk.com/wall-24234717_2297...,0
1,1,Как вступить в профсоюз,Как вступить в профсоюз? Чтобы вступить в Проф...,1


link to model in HuggingFace [e5-base-en-ru](https://huggingface.co/d0rj/e5-base-en-ru)

In [7]:
tokenizer = XLMRobertaTokenizer.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)
search_model = XLMRobertaModel.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/471 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/529M [00:00<?, ?B/s]

# 1. Создание БД c помощью e5


In [10]:
all_chunks = []

for answer, topic_name in zip(answers['answer'], answers['topic_name']):
    all_chunks.append(Document(
        page_content=answer,
        metadata={
            "source": topic_name
        }
    ))

# Инициализация эмбеддера E5
embedder = E5LangChainEmbedder(
    tokenizer=tokenizer,
    model=search_model,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    add_prefix=True,  #!!!
    disable_tqdm=False,
)

# Создание или загрузка векторного хранилища Chroma
vector_store = Chroma(
    collection_name="docs",
    embedding_function=embedder,
    persist_directory="./chroma_db"  #!!!
)

# Безопасное добавление документов в векторное хранилище
safe_add_documents(vector_store, all_chunks)

Добавление в Chroma: 100%|██████████| 105/105 [01:43<00:00,  1.02doc/s]

Все документы успешно добавлены!


In [11]:
!zip -r chroma_db.zip chroma_db/

  adding: chroma_db/ (stored 0%)
  adding: chroma_db/2ff8dfc9-dbef-45a9-ba56-0ad19cc6264b/ (stored 0%)
  adding: chroma_db/2ff8dfc9-dbef-45a9-ba56-0ad19cc6264b/link_lists.bin (stored 0%)
  adding: chroma_db/2ff8dfc9-dbef-45a9-ba56-0ad19cc6264b/header.bin (deflated 61%)
  adding: chroma_db/2ff8dfc9-dbef-45a9-ba56-0ad19cc6264b/length.bin (deflated 100%)
  adding: chroma_db/2ff8dfc9-dbef-45a9-ba56-0ad19cc6264b/data_level0.bin (deflated 100%)
  adding: chroma_db/chroma.sqlite3 (deflated 52%)


# Подключение БД

In [12]:
vector_store = Chroma(
    collection_name="docs",
    embedding_function=embedder,
    persist_directory="./chroma_db"
)

In [15]:
query = "Как поступить попасть в профсоюз?"

relevant_docs = vector_store.similarity_search(
    query,
    k=3,
)

In [16]:
relevant_docs

[Document(id='b1d2ef02-3992-4e90-be9c-a8aecd32505e', metadata={'source': 'Как вступить в профсоюз'}, page_content='Как вступить в профсоюз? Чтобы вступить в Профсоюз, можно прийти в кабинет Профкома (2-39) в рабочие часы (11:00-16:00) или воспользоваться сервисом в приложении «Твой ФФ» и подать заявление на сайте. Подписанное заявление также необходимо отнести в кабинет Профкома.'),
 Document(id='5dd77e06-7148-439d-95ab-3bcbcba74ab4', metadata={'source': 'Вступление в профсоюз'}, page_content='Вступление в профсоюз. Как вступить в Профсоюз? Чтобы вступить в Профсоюз, можно прийти в кабинет Профкома (2-39) в рабочие часы (11:00-16:00) или воспользоваться сервисом в приложении «Твой ФФ» и подать заявление на сайте. Подписанное заявление также необходимо отнести в кабинет Профкома. Члены Профсоюза платят членские взносы в размере 4% от стипендии (для студентов, обучающихся на бюджете) или 240 рублей в год (при обучении на контрактной основе).'),
 Document(id='cc00ff71-c9fc-4591-a7e5-7a200